### Stochastic differential equations

Simulate Geometric Brownian motion using SDE (Stochastic differential equations) approach<br>
$$
dS_t=rS_tdt + \sigma S_t dW_t
$$
On a whiteboard derive expectation $\mathsf{E}(S_t).$

In [ ]:
import numpy as np
from scipy.stats import norm, linregress
import plotly.graph_objects as go

In [ ]:
r = 0.15
sigma = np.sqrt(0.4)
S0 = 100
print(r-sigma**2/2)

T = 2
m = 100
h = 1/100
N = 25_000

t = np.linspace(0, T, T * m + 1)

paths = []
for i in range(N):
    path = [S0]
    ksi = norm.rvs(0, h ** 0.5, m * T)
    
    for j in range(1, T*m + 1):
        val = path[-1] + r * path[-1] * h + sigma * ksi[j - 1] * path[-1]
        path.append(val)
    paths.append(np.array(path))
    
# plot = go.Figure()

# for path in paths:
#     graph = go.Scatter(x = t, y = path)
#     plot.add_trace(graph)
# plot.show()

Verify $E(S_2)=S_0 e^{r\cdot 2}$

In [ ]:
paths_np = np.array(paths)
theor_mean = S0*np.exp(2*r)
S2 = paths_np[:, -1]
mean = np.mean(S2)
print(mean, abs(mean-theor_mean))

### Bachelier model

In [ ]:
import numpy as np
import pandas as pd

import random

import plotly.graph_objects as go

https://docs.python.org/3/library/random.html

pip install yahooquery

Yahooquery documentation:<br>
https://yahooquery.dpguthrie.com/<br>
https://yahooquery.dpguthrie.com/guide/ticker/historical/

In [ ]:
from yahooquery import Ticker

In [ ]:
cola = Ticker('KO')
cola_global_data = cola.history(period='max', interval='1d')
cola_global_data

In [ ]:
global_prices = cola_global_data['close']

In [ ]:
cola_global_data.columns
cola_global_data.index

In [ ]:
t = cola_global_data.index.get_level_values(level=1)

plot = go.Figure()
graph = go.Scatter(x = t, y = global_prices)
plot.add_trace(graph)
plot.show()

Analyse daily data from 2010 to 2022

In [ ]:
cola_data = cola.history(start = '2010-01-01', end = '2022-12-31')
t = cola_data.index.get_level_values(level=1)
prices = cola_data['close']

In [ ]:
plot = go.Figure()
graph = go.Scatter(x = t, y = prices)
plot.add_trace(graph)
plot.show()

To check how models work choose and analyse trajectory Feb 19, 2019 - Feb 21, 2020

In [ ]:
cola_data = cola.history(start = '2019-02-19', end = '2020-02-21')
t = cola_data.index.get_level_values(level=1)
prices = cola_data['close']

In [ ]:
plot = go.Figure()
graph = go.Scatter(x = t, y = prices)
plot.add_trace(graph)
plot.show()

In [ ]:
# cola_data = pd.read_csv('Topic-5_cola_data.csv')
# cola_data['close']

<H3>Bachelier model</H3> - first serious model in mathematical finance. It assumes that stock prices $(S_t)_{t\geq 0}$ satisfy equation
$$S_t=S_0+\mu t+\sigma W_t$$
$\mu$ is called <i>trend</i>, $\sigma$ is called <i>volatility</i>.

In [ ]:
def bm_simulations(n_paths, granularity, max_time):
    n_points = granularity * max_time
    
    paths = []
    for _ in range(n_paths):
        xs = [0] + random.choices([-1, 1], weights = [0.5, 0.5], k = n_points)
        path = np.cumsum(xs)/np.sqrt(granularity)
        paths.append(path)
    
    return np.array(paths)

<b>Step 1.</b> Construct Brownian motion.

In [ ]:
s0 = prices.iloc[0]

N = 7
T = len(t)
T
gran_factor = 10
W = bm_simulations(N, gran_factor, T) 

Modify simulated prices to be able to draw prices together with simulations in one graph correctly.

In [ ]:
n_pts = T*gran_factor+1
t_ext = np.linspace(0, T, n_pts)
t_num = np.arange(0, T)len(t_num) == len(prices)

In [ ]:
plot = go.Figure()

graph = go.Scatter(x = t_num, y = prices, name = 'Prices')
plot.add_trace(graph)
for i in range(0, N):
    graph1 = go.Scatter(x = t_ext, y = s0+W[i], name = f's0+W[{i}]')
    plot.add_trace(graph1)
plot.show()

<H3>Step 2. Parameters calibration.</H3>
Essentially it is an estimation of <i>trend</i> $\mu$ and <i>volatility</i> $\sigma$.

Estimate $\mu$ via linear regression $y=\mu x+\varepsilon$ with $x=t,\,y=S_t=prices[t]$

In [ ]:
ans = linregress(t_num, prices)
mu = ans.slope
mu

Check that regression has been implemented correctly

Build intermediate result $S_t = S_0+\hat{\mu} t+W_t$

In [ ]:
plot = go.Figure()

graph = go.Scatter(x = t_num, y = prices, name = 'Prices')
plot.add_trace(graph)

for i in range(0, N):
    graph1 = go.Scatter(x = t_ext, y = s0+mu*t_ext+W[i], name = f'BM with drift {i}')
    plot.add_trace(graph1)
plot.show()

Volatility estimation: should we take standard deviation = deviation from average price or something else?

Alternative estimate:
$$\hat{\sigma}_{reg}^2 = \frac{1}{N+1}\sum_{i=0}^{N} (S_{t_i}-\mu t_i-S_0)^2$$

In [ ]:
deltas_sq = (prices - mu*t_num - s0)**2
sigma_reg = np.sqrt(np.mean(deltas_sq))
sigma_reg

Seems that volatility is a bit high. Try $\chi^2$ approach: relative errors 
$$
\chi^2_{st} = \frac{1}{n+1}\sum_{i=0}^n \frac{(Obs_i-Exp_i)^2}{Exp_i}
$$
credit: @Alexander Polyakov

In [ ]:
rel_deltas = (prices - s0 - mu*t_num)**2/(s0+mu*t_num)
sigma_chi = np.sqrt(np.mean(rel_deltas))
sigma_chi

In [ ]:
plot = go.Figure()

graph = go.Scatter(x = t_num, y = prices, name = 'Prices')
plot.add_trace(graph)

for i in range(0, N):
    graph1 = go.Scatter(x = t_ext, y = s0+mu*t_ext+sigma_chi*W[i], name = f'Bachelier {i}')
    plot.add_trace(graph1)
plot.show()